In [0]:
dbutils.widgets.text("catalog", "dev")
dbutils.widgets.text("schema", "bikeshare")
dbutils.widgets.text("volume", "raw")
dbutils.widgets.text("feeds", "station_status")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")
feeds_wanted = [f.strip() for f in dbutils.widgets.get("feeds").split(",") if f.strip()]

VOL = f"/Volumes/{catalog}/{schema}/{volume}"
print(VOL, feeds_wanted)

In [0]:
import time
import requests

DISCOVERY_URL = "https://sharedmobility.ch/gbfs.json"   # GBFS 2.0, no key needed

def get_json(url, attempts=4, wait_s=20):
    headers = {"Cache-Control": "no-cache"}
    for i in range(attempts):
        r = None
        try:
            r = requests.get(url, headers=headers, timeout=(10, 60), stream=True)
            r.raise_for_status()
            return r.json()          # the body is read here, so a cut-off download raises here
        except (requests.exceptions.RequestException, ValueError) as e:
            print(f"Attempt {i + 1}/{attempts} failed for {url}: {e}")
            if r is not None:
                keep = ("content-length", "content-encoding", "transfer-encoding",
                        "age", "via", "x-cache", "server", "date", "last-modified")
                print({k: v for k, v in r.headers.items() if k.lower() in keep})
            if i == attempts - 1:
                raise
            time.sleep(wait_s * (i + 1))

disc = get_json(DISCOVERY_URL)
lang = next(iter(disc["data"]))
feeds = {f["name"]: f["url"] for f in disc["data"][lang]["feeds"]}

unknown = [f for f in feeds_wanted if f not in feeds]
if unknown:
    raise ValueError(f"Unknown feeds: {unknown}. Available: {sorted(feeds)}")
feeds

In [0]:
import json
from datetime import datetime, timezone

ts = datetime.now(timezone.utc).strftime("%Y-%m-%d-%H-%M-%SZ")
print(f"Timestamp: {ts}")

for name in feeds_wanted:
    payload = get_json(feeds[name])
    folder = f"{VOL}/{name}"
    dbutils.fs.mkdirs(folder)
    path = f"{folder}/{name}_{ts}.json"
    with open(path, "w") as fh:
        json.dump(payload, fh)
    n_items = {k: len(v) for k, v in payload["data"].items() if isinstance(v, list)}
    print(name, n_items, "->", path)